# conditional-hparam-branch — ex1: Linear with optional bias gated by use_bias flag

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conditional-hparam-branch`. Running the final beacon cell reports progress against the `PyTorch: Conditional hparam branch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Conditional hparam branch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conditional-hparam-branch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conditional-hparam-branch"
DD_SUBTOPIC = "PyTorch: Conditional hparam branch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conditional hparam branch — quick refresher

Many `nn.Module` configs have an optional component — bias, dropout, layer-norm — gated by a boolean hyperparameter. The canonical pattern uses `if` inside `__init__` to either register the sub-component as a Parameter/Module or set the slot to `None`:

```python
class Block(nn.Module):
    def __init__(self, dim, use_bias=True, dropout=0.0):
        super().__init__()
        self.linear = nn.Linear(dim, dim, bias=use_bias)
        if dropout > 0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = None
    def forward(self, x):
        x = self.linear(x)
        if self.dropout is not None:
            x = self.dropout(x)
        return x
```

**Why set to `None`, not just skip the assignment.** A consistent slot name (`self.dropout`) lets `forward` always reference it; the `is not None` guard then short-circuits cleanly. Skipping the assignment would raise `AttributeError` on the `forward` branch, which is the worst kind of runtime failure — bypasses every static-analysis tool.

**`nn.Identity()` is the alternative.** `self.dropout = nn.Dropout(p) if p > 0 else nn.Identity()` gives you an always-callable slot — `forward` becomes `x = self.dropout(x)` unconditionally. Pick this when the conditional branch is hot (avoid the per-call `is not None` check) and the no-op sub-module's tiny overhead is fine.

### Exercise 1 — Linear with optional bias gated by use_bias flag

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Define a Module whose `__init__` registers `self.bias` as an nn.Parameter when use_bias=True and as `None` otherwise, with the matching forward-pass `is not None` branch.
> Keywords: bias, optional, conditional, if-branch, use_bias
> ```

**KCs targeted:** `conditional-param-register`, `forward-branch-on-none`

Implement `OptionalBiasLinear` — a Linear-style Module where the bias is optional based on a constructor flag:

1. Class `OptionalBiasLinear(t.nn.Module)`:
   - `__init__(self, in_features, out_features, use_bias=True)`:
     - Call `super().__init__()`.
     - Store `self.in_features`, `self.out_features`, `self.use_bias`.
     - Create `self.weight = t.nn.Parameter(t.zeros(out_features, in_features))` (zero-init so the math is predictable in tests).
     - **Conditional branch on `use_bias`:**
       - If `use_bias`: `self.bias = t.nn.Parameter(t.zeros(out_features))`.
       - Else: `self.bias = None`.  # NOT `del` or skipped
   - `forward(self, x: Tensor) -> Tensor`:
     - `out = x @ self.weight.T`
     - **Conditional branch:** `if self.bias is not None: out = out + self.bias`.
     - Return `out`.
2. Return an instance from `ex1_optional_bias_linear(in_features, out_features, use_bias)`.

**The two halves are coupled.** Setting `self.bias = None` (rather than skipping the assignment) means `forward` can ALWAYS reference `self.bias` — the `is not None` guard then handles the absent case. If you skipped the assignment, the forward branch would raise `AttributeError` at the worst possible time (mid-training).

The test asserts: (a) with `use_bias=True`, both weight and bias show up in `.parameters()`; (b) with `use_bias=False`, only weight appears AND `self.bias is None`; (c) forward output matches the right algebraic expression in both modes.

In [ ]:
def ex1_optional_bias_linear(in_features: int, out_features: int, use_bias: bool):
    class OptionalBiasLinear(t.nn.Module):
        def __init__(self, in_features, out_features, use_bias=True):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            self.use_bias = use_bias
            self.weight = t.nn.Parameter(t.zeros(out_features, in_features))
            if use_bias:
                self.bias = t.nn.Parameter(t.zeros(out_features))
            else:
                self.bias = None
        def forward(self, x: Tensor) -> Tensor:
            out = x @ self.weight.T
            if self.bias is not None:
                out = out + self.bias
            return out
    return OptionalBiasLinear(in_features, out_features, use_bias)


<details><summary>Solution</summary>

```python
def ex1_optional_bias_linear(in_features: int, out_features: int, use_bias: bool):
    class OptionalBiasLinear(t.nn.Module):
        def __init__(self, in_features, out_features, use_bias=True):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            self.use_bias = use_bias
            self.weight = t.nn.Parameter(t.zeros(out_features, in_features))
            if use_bias:
                self.bias = t.nn.Parameter(t.zeros(out_features))
            else:
                self.bias = None
        def forward(self, x: Tensor) -> Tensor:
            out = x @ self.weight.T
            if self.bias is not None:
                out = out + self.bias
            return out
    return OptionalBiasLinear(in_features, out_features, use_bias)
```

**Why `self.bias = None` and not skipping the assignment.** `nn.Module.__setattr__` has a special case for `None` — setting a Parameter slot to `None` un-registers it (or never registers it) without raising. That's why `mod_without.bias` returns `None` cleanly and `forward`'s `is not None` guard works.

**Why `self.bias is not None` and not `self.bias`.** Calling `bool(tensor)` on a tensor with multiple elements raises `RuntimeError: Boolean value of Tensor with more than one element is ambiguous`. The `is not None` check sidesteps the ambiguity — it's the safe idiom and what `nn.Linear.forward` actually does.

**`nn.Identity()` as the alternative.** If the conditional branch is on a hot path (every forward) you can avoid the `is not None` check by assigning `self.dropout = nn.Dropout(p) if p > 0 else nn.Identity()`. Then `forward` is always `x = self.dropout(x)` — `Identity.forward(x)` just returns `x`. Tradeoff: a tiny per-call overhead for the no-op Module vs. a branch in Python.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()